# Pipeline stage visualization

Static replacement for the former `debug_app.py` Tk GUI. Runs `Pipeline` **once**
on a single page, then for every stage renders:

- one **original page** image, then
- for each overlay category of that stage, a pair: the category's geometry drawn
  alone on white (**isolated**), and the original page with that geometry drawn on
  top (**overlay**).

So a single-overlay stage produces 3 images; a stage with *N* overlays produces
`1 + 2N`. `clustering` and `color_separation` expand **every** category / bucket,
which can be 100+ images on a dense page.


In [ ]:
import sys, io, colorsys, hashlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "pipeline_stage_visualization.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pymupdf as fitz
from PIL import Image, ImageDraw

from rastervec.helpers.geometry import union_bbox
from rastervec.logging_setup import configure_logging
from rastervec.pipeline import SPATIAL_REGROUP_TOLERANCE_PX, Pipeline, PipelineContext
from rastervec.Reader.reader import Reader
from rastervec.renderer import Renderer
from rastervec.Vector_Classification.classification import SIGNATURE_ROUND_PX
from rastervec.Vector_Classification.items import item_filters as vc

configure_logging()
RENDERER = Renderer()


## Parameters

In [ ]:
PDF_PATH = next(iter(sorted((PROJECT_ROOT / "references").glob("*.pdf"))), None)
PAGE_INDEX = 0
ZOOM = 2.0
FINAL_STAGE = None  # e.g. "unique_clusters" -> skip the FAST weights file + PaddleOCR

assert PDF_PATH is not None, "no PDF under references/ -- set PDF_PATH by hand"
print("PDF:", PDF_PATH, "| page", PAGE_INDEX)


## Run the pipeline

In [ ]:
reader = Reader(str(PDF_PATH))
ctx = PipelineContext(reader=reader, page_index=PAGE_INDEX)
# _run_stages mutates ctx AND returns the per-stage StageOutput list -- one run,
# both the accumulated ctx.* fields and each stage's status/error.
outputs = {o.key: o for o in Pipeline._run_stages(ctx, FINAL_STAGE)}
page = ctx.page
assert page is not None, outputs["reader"].error

for key, o in outputs.items():
    print(f"{'ok ' if o.status == 'ok' else 'ERR'}  {key:18}  {o.error or ''}")


## Rendering helpers

In [ ]:
CLUSTER_STEP_COLORS = ["#2563eb", "#7c3aed", "#ea580c", "#0d9488", "#6b7280", "#c026d3", "#65a30d", "#0284c7"]
SIDE_CATEGORY_COLORS = ["#9ca3af", "#f59e0b", "#db2777", "#eab308", "#16a34a"]
SIMILARITY_GROUP_COLORS = ["#059669", "#2563eb", "#dc2626", "#d97706", "#7c3aed", "#0891b2", "#be185d", "#65a30d"]
OCR_PASSED_COLOR = "#059669"
OCR_FAILED_COLOR = "#dc2626"
NATIVE_WORD_COLOR = "#2563eb"
SPATIAL_REGROUP_COLOR = "#8b5cf6"
ROT_APPLIED_COLOR = "#f97316"
ROT_SKIPPED_COLOR = "#94a3b8"
DEFAULT_PATH_COLOR = "#111827"


def display_matrix(zoom=None):
    """page (unrotated MediaBox) space -> raster-pixel space, page rotation baked
    in -- the rule the deleted debug_app._get_display_matrix used."""
    z = ZOOM if zoom is None else zoom
    return page.fitz_page.rotation_matrix * fitz.Matrix(z, z)


def page_raster(zoom=None):
    z = ZOOM if zoom is None else zoom
    pix = page.fitz_page.get_pixmap(matrix=fitz.Matrix(z, z))
    return Image.open(io.BytesIO(pix.pil_tobytes(format="PNG"))).convert("RGB")


ORIGINAL = page_raster()


def _xf(x, y, m):
    p = fitz.Point(x, y) * m
    return (p.x, p.y)


def draw_paths(img, paths, color=None, width=2):
    """Port of debug_app._draw_vector_path: polyline of path.points (curves as
    straight segments through control points, exactly as the Tk canvas drew
    them). color=None -> each path in its own real PDF stroke/fill colour."""
    m = display_matrix()
    d = ImageDraw.Draw(img)
    for p in paths:
        pts = [_xf(x, y, m) for x, y in p.points]
        if len(pts) < 2:
            continue
        c = color or RENDERER.path_color_hex(p)
        if p.kind in ("re", "qu"):
            d.polygon(pts, outline=c, width=width)
        else:
            d.line(pts, fill=c, width=width)


def draw_polys(img, polys, color, width=2):
    m = display_matrix()
    d = ImageDraw.Draw(img)
    for poly in polys:
        pts = [_xf(x, y, m) for x, y in poly]
        if len(pts) >= 2:
            d.polygon(pts, outline=color, width=width)


def draw_bboxes(img, bboxes, color, width=2):
    m = display_matrix()
    d = ImageDraw.Draw(img)
    for bb in bboxes:
        x0, y0 = _xf(bb[0], bb[1], m)
        x1, y1 = _xf(bb[2], bb[3], m)
        d.rectangle([min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)], outline=color, width=width)


def blank_like(img):
    return Image.new("RGB", img.size, "white")


def show_row(images, titles, height=5.0):
    n = len(images)
    ar = images[0].height / images[0].width
    fig, axes = plt.subplots(1, n, figsize=(max(height / ar * n, 4.0), height))
    axes = [axes] if n == 1 else list(axes)
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im)
        ax.set_title(t, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def _paint(img, cat):
    if cat.get("paths"):
        draw_paths(img, cat["paths"], cat.get("path_color"), cat.get("width", 2))
    if cat.get("polys"):
        draw_polys(img, cat["polys"], cat.get("color", DEFAULT_PATH_COLOR), cat.get("width", 2))
    if cat.get("bboxes"):
        draw_bboxes(img, cat["bboxes"], cat.get("color", DEFAULT_PATH_COLOR), cat.get("width", 2))


def _iso_and_overlay(cat):
    iso, ovl = cat.get("isolated"), cat.get("overlay")
    if iso is not None:
        iso = iso.convert("RGB")
    else:
        iso = blank_like(ORIGINAL)
        _paint(iso, cat)
    if ovl is not None:
        ovl = ovl.convert("RGB")
    else:
        ovl = ORIGINAL.copy()
        _paint(ovl, cat)
    return iso, ovl


def visualize(stage_key, categories, note=None):
    out = outputs.get(stage_key)
    print(f"=== {out.label if out else stage_key}  ({stage_key}) ===")
    if out is None:
        print("  (stage not run -- FINAL_STAGE stopped the pipeline earlier)")
        return
    if out.status == "error":
        print(f"  ERROR: {out.error}")
        return
    if note:
        print(" ", note)
    show_row([ORIGINAL], ["original page"])
    for cat in categories:
        iso, ovl = _iso_and_overlay(cat)
        show_row([iso, ovl], [f"{cat['name']} -- isolated", f"{cat['name']} -- overlay"])
    if not categories:
        print("  (no overlays for this stage)")


def signature_color(sig):
    digest = hashlib.md5(repr(sig).encode()).hexdigest()
    hue = (int(digest[:8], 16) % 360) / 360.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.65, 0.85)
    return "#%02x%02x%02x" % (round(r * 255), round(g * 255), round(b * 255))


def fast_mask_overlay(base, mask):
    heat = (np.clip(mask, 0.0, 1.0) * 255).astype("uint8")
    zeros = Image.new("L", base.size, 0)
    heat_img = Image.merge("RGB", (Image.fromarray(heat), zeros, zeros))
    return Image.blend(base.convert("RGB"), heat_img, alpha=0.5)


## 1. Reader

In [ ]:
visualize("reader", [], note=f"mediabox={page.meta.mediabox}  rotation={page.meta.rotation}  size={page.meta.width:.0f}x{page.meta.height:.0f}")


## 2. Native Text

In [ ]:
words = ctx.native_words or []
visualize("native", [{
    "name": f"text words ({len(words)})",
    "color": NATIVE_WORD_COLOR,
    "polys": [w.quad for w in words],
    "isolated": RENDERER.render_reconstructed_page(page.meta, native_words=words, zoom=ZOOM),
}])


## 3. Vector Extraction

In [ ]:
vp = ctx.vector_paths or []
kinds = sorted({p.kind for p in vp})
visualize("vector_extract", [
    {"name": f"kind {k!r} ({sum(p.kind == k for p in vp)})", "paths": [p for p in vp if p.kind == k]}
    for k in kinds
], note=f"{len(vp)} paths, kinds={kinds}")


## 4. Layer Separation

In [ ]:
pbl = ctx.paths_by_layer or {}
visualize("layer_separation", [
    {"name": f"{name or '(no layer)'} ({len(ps)})", "paths": ps}
    for name, ps in pbl.items()
], note=f"{len(pbl)} layer(s)")


## 5. Color Separation

In [ ]:
pblc = ctx.paths_by_layer_color or {}
cats = []
for layer, by_color in pblc.items():
    for color, ps in by_color.items():
        cats.append({"name": f"{layer or '(no layer)'} / {color} ({len(ps)})", "paths": ps})
visualize("color_separation", cats, note=f"{len(cats)} (layer, color) bucket(s)")


## 6. Clustering

Every classification step's every category (one `kept` per step + any `dropped` /
`info` side categories), merged across all `(layer, color)` buckets. `kept` bboxes
use a per-step colour; side categories cycle `SIDE_CATEGORY_COLORS`. A final
"colour by vector type" image hashes each post-step-1 path's shape signature.


In [ ]:
clustering = ctx.clustering or {}
results = list(clustering.values())
step_labels = [s.label for s in results[0].steps] if results else []
cats = []
for i, label in enumerate(step_labels):
    names = []
    for r in results:
        for nm in r.steps[i].categories:
            if nm not in names:
                names.append(nm)
    side_i = 0
    for nm in names:
        groups, role = [], "kept"
        for r in results:
            cat = r.steps[i].categories.get(nm)
            if cat is None:
                continue
            role = cat.role
            groups.extend(g for g in cat.groups if g)
        if role == "kept":
            color = CLUSTER_STEP_COLORS[i % len(CLUSTER_STEP_COLORS)]
        else:
            color = SIDE_CATEGORY_COLORS[side_i % len(SIDE_CATEGORY_COLORS)]
            side_i += 1
        cats.append({
            "name": f"step {i + 1} {label} / {nm} [{role}] ({len(groups)} grp)",
            "color": color,
            "bboxes": [union_bbox([p.bbox for p in g]) for g in groups],
        })

sig_paths = [
    p for r in results for s in r.steps
    if s.signature_counts is not None
    for g in s.categories["kept"].groups for p in g
]
if sig_paths:
    iso, ovl = blank_like(ORIGINAL), ORIGINAL.copy()
    m = display_matrix()
    for im in (iso, ovl):
        d = ImageDraw.Draw(im)
        for p in sig_paths:
            pts = [_xf(x, y, m) for x, y in p.points]
            if len(pts) >= 2:
                d.line(pts, fill=signature_color(vc.vector_signature(p, SIGNATURE_ROUND_PX)), width=2)
    cats.append({"name": "colour by vector type", "isolated": iso, "overlay": ovl})

visualize("clustering", cats, note=f"{len(results)} (layer, color) bucket(s), {len(step_labels)} steps")


## 7. Text Candidates

In [ ]:
tc = ctx.text_clusters or []
visualize("text_candidates", [{
    "name": f"text candidate clusters ({len(tc)})",
    "color": CLUSTER_STEP_COLORS[0],
    "bboxes": [union_bbox([p.bbox for p in c]) for c in tc if c],
    "paths": [p for c in tc for p in c],
    "path_color": CLUSTER_STEP_COLORS[0],
}])


## 8. Unique Clusters

In [ ]:
sg = ctx.similarity_groups or []
visualize("unique_clusters", [
    {
        "name": f"similarity group {gi + 1} ({len(grp)} cluster(s))",
        "color": SIMILARITY_GROUP_COLORS[gi % len(SIMILARITY_GROUP_COLORS)],
        "bboxes": [union_bbox([p.bbox for p in c]) for c in grp if c],
    }
    for gi, grp in enumerate(sg)
], note=f"{len(sg)} similarity group(s)")


## 9. FAST: Text Detect

In [ ]:
fr = ctx.fast_result
_fast_out = outputs.get("fast_text_detect")
if _fast_out is None or _fast_out.status == "error":
    visualize("fast_text_detect", [])
elif fr is None or fr.page_image is None:
    print("=== FAST: Text Detect ===\n  (no vector paths on this page)")
else:
    render = fr.page_image.convert("RGB")
    heat = fast_mask_overlay(render, fr.page_mask) if fr.page_mask is not None else render
    passed, dropped = ctx.fast_passed or [], ctx.fast_dropped or []
    print(f"  detect_seconds = {fr.detect_seconds}")
    for c in passed + dropped:
        print(f"  score {fr.scores.get(id(c), 0.0) * 100:5.1f}%  {'PASS' if c in passed else 'drop'}  {len(c)} path(s)")
    visualize("fast_text_detect", [
        {"name": "FAST render", "isolated": render, "overlay": render},
        {"name": "detection heatmap", "isolated": heat, "overlay": heat},
        {"name": f"passed clusters ({len(passed)})", "color": OCR_PASSED_COLOR,
         "bboxes": [union_bbox([p.bbox for p in c]) for c in passed if c]},
        {"name": f"dropped clusters ({len(dropped)})", "color": OCR_FAILED_COLOR,
         "bboxes": [union_bbox([p.bbox for p in c]) for c in dropped if c]},
    ])


## 10. Spatial Regroup

In [ ]:
rg = ctx.regrouped_clusters or []
visualize("spatial_regroup", [{
    "name": f"regrouped clusters ({len(rg)})",
    "color": SPATIAL_REGROUP_COLOR,
    "bboxes": [union_bbox([p.bbox for p in c]) for c in rg if c],
    "paths": [p for c in rg for p in c],
    "path_color": SPATIAL_REGROUP_COLOR,
}], note=f"{len(ctx.fast_passed or [])} FAST-passed -> {len(rg)} regrouped (tol={SPATIAL_REGROUP_TOLERANCE_PX}px)")


## 11. OCR Compare

In [ ]:
cor = ctx.cluster_ocr_results or []
passed = [r for r in cor if r.resolved.text.strip()]
failed = [r for r in cor if not r.resolved.text.strip()]
for r in passed:
    print(f"  {r.resolved.text!r:42}  conf={r.resolved.confidence:.2f}  rot={r.resolved.rotation_used:>3}  {r.ocr_seconds:.2f}s")
visualize("ocr_compare", [
    {"name": f"passed ({len(passed)})", "color": OCR_PASSED_COLOR,
     "bboxes": [r.resolved.bbox for r in passed],
     "isolated": RENDERER.render_reconstructed_page(page.meta, ocr_results=[r.resolved for r in passed], zoom=ZOOM)},
    {"name": f"failed ({len(failed)})", "color": OCR_FAILED_COLOR,
     "bboxes": [r.resolved.bbox for r in failed],
     "paths": [p for r in failed for p in r.cluster], "path_color": OCR_FAILED_COLOR},
])


## 12. Rotation Verify

In [ ]:
rc = ctx.rotation_checks or []
applied = [c for c in rc if c.applied]
skipped = [c for c in rc if not c.applied]
for c in rc:
    print(f"  {c.text!r:42}  {c.before_rotation:>3} -> {c.after_rotation:>3}  err {c.error_unrotated:.2f}/{c.error_rotated:.2f}  {'FIX' if c.applied else ''}")
visualize("rotation_verify", [
    {"name": f"applied ({len(applied)})", "color": ROT_APPLIED_COLOR, "width": 3,
     "bboxes": [c.bbox for c in applied],
     "isolated": RENDERER.render_reconstructed_page(page.meta, ocr_results=[c.resolved for c in rc if c.resolved.text.strip()], zoom=ZOOM)},
    {"name": f"not applied ({len(skipped)})", "color": ROT_SKIPPED_COLOR,
     "bboxes": [c.bbox for c in skipped]},
])


## 13. Drawing Vectors

In [ ]:
dv = ctx.drawing_vectors or []
dashed = [d for d in dv if d.dashed]
solid = [d for d in dv if not d.dashed]
_recon = RENDERER.render_reconstructed_page(page.meta, drawing_vectors=dv, zoom=ZOOM)
visualize("drawing_vectors", [
    {"name": f"dashed ({len(dashed)})", "color": DEFAULT_PATH_COLOR, "bboxes": [d.bbox for d in dashed]},
    {"name": f"solid ({len(solid)})", "color": DEFAULT_PATH_COLOR, "bboxes": [d.bbox for d in solid]},
    {"name": "full reconstruction", "isolated": _recon, "overlay": _recon},
], note=f"{len(dv)} drawing vector(s)")


## Reading the results

- Everything a classification step **drops** (`clustering` side categories,
  `fast_text_detect` dropped, `ocr_compare` failed) is folded into
  `drawing_vectors` -- it's the pipeline's "this is drawing content, not text"
  verdict.
- `ocr_compare` treats a **blank** reading as a failure; a non-blank reading is
  what survives into the final output.
- `rotation_verify` only *flags* a 90 deg correction (`applied`); it never mutates
  `ocr_compare`'s own reading -- the corrected orientation lives on
  `RotationCheck.resolved`, which the `rotation_verify` / `drawing_vectors`
  reconstructions read.
- Set `FINAL_STAGE` before `fast_text_detect` to run without the FAST weights
  file, or before `ocr_compare` to skip building PaddleOCR.
